# Checks

Run All after every change in `src/netad/`. Stops at the first failing assert.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd

from netad.data import (
    DATA_DIR, FEATURES, CATEGORICAL, NUMERIC, TARGETS, EXPECTED_COLUMNS,
    load_unsw_frame, load_unsw, deduplicate, resolve_conflicts, overlap_with_train,
    train_val_split,
)
from netad.preprocessing import build_preprocessor

## data.py

### Schema

In [3]:
assert DATA_DIR.is_dir(), DATA_DIR
assert len(CATEGORICAL) == 3
assert len(NUMERIC) == 39
assert len(FEATURES) == 42
assert set(CATEGORICAL).isdisjoint(NUMERIC)
assert len(set(EXPECTED_COLUMNS)) == len(EXPECTED_COLUMNS) == 45

### load_unsw_frame

In [4]:
train_df = load_unsw_frame("train")
test_df = load_unsw_frame("test")

assert train_df.shape == (175341, 45), train_df.shape
assert test_df.shape == (82332, 45), test_df.shape
assert train_df.columns[0] == "id", train_df.columns[0]
assert set(train_df.columns) == set(test_df.columns) == set(EXPECTED_COLUMNS)
assert train_df.isna().sum().sum() == 0
assert test_df.isna().sum().sum() == 0

### load_unsw

In [5]:
X_train, y_train, y_train_multi = load_unsw("train")
X_test, y_test, y_test_multi = load_unsw("test")

assert X_train.shape == (175341, 42), X_train.shape
assert X_test.shape == (82332, 42), X_test.shape
assert list(X_train.columns) == FEATURES
assert not {"id", "label", "attack_cat"} & set(X_train.columns)
assert set(y_train.unique()) == set(y_test.unique()) == {0, 1}
assert ((y_train_multi == "Normal") == (y_train == 0)).all()
assert ((y_test_multi == "Normal") == (y_test == 0)).all()
assert y_train_multi.nunique() == y_test_multi.nunique() == 10
assert all(pd.api.types.is_string_dtype(X_train[c]) for c in CATEGORICAL)
assert all(pd.api.types.is_numeric_dtype(X_train[c]) for c in NUMERIC)

### deduplicate

In [6]:
dedup = deduplicate(train_df)

assert len(dedup) == 107740, len(dedup)
assert not dedup.duplicated(FEATURES + TARGETS).any()
assert dedup.index.isin(train_df.index).all()
assert len(train_df) == 175341

### resolve_conflicts

Row counts after `resolve_conflicts` + `deduplicate`.

In [7]:
expected = {
    ("label", "drop"): 107282,
    ("label", "majority"): 107412,
    ("attack_cat", "drop"): 99268,
    ("attack_cat", "majority"): 100427,
}

for (target, policy), n_rows in expected.items():
    resolved = resolve_conflicts(train_df, target, policy)
    assert resolved.index.isin(train_df.index).all()
    # one target value per feature vector
    assert len(resolved.drop_duplicates(FEATURES)) == len(resolved.drop_duplicates(FEATURES + [target])), (target, policy)
    assert len(deduplicate(resolved)) == n_rows, (target, policy, len(deduplicate(resolved)))

In [8]:
assert len(resolve_conflicts(train_df)) == len(resolve_conflicts(train_df, "attack_cat", "majority"))

try:
    resolve_conflicts(train_df, policy="random")
except ValueError:
    pass
else:
    raise AssertionError("bad policy should raise ValueError")

### overlap_with_train

In [9]:
in_train = overlap_with_train(train_df, test_df)

assert in_train.dtype == bool
assert in_train.index.equals(test_df.index)
assert in_train.sum() == 8541, in_train.sum()

### Errors

In [10]:
try:
    load_unsw("validation")
except ValueError:
    pass
else:
    raise AssertionError("bad split should raise ValueError")

try:
    load_unsw("train", data_dir="C:/nu_exista")
except FileNotFoundError:
    pass
else:
    raise AssertionError("missing file should raise FileNotFoundError")

### train_val_split

Rows before the split: train cleaned for the target, one row per feature vector.

In [11]:
splits = {}
for target, n_rows in [("label", 100941), ("attack_cat", 100427)]:
    X_tr, X_val, y_tr, y_val = train_val_split(train_df, target)
    splits[target] = X_tr, X_val, y_tr, y_val

    assert len(X_tr) + len(X_val) == n_rows, (target, len(X_tr) + len(X_val))
    assert abs(len(X_val) / n_rows - 0.2) < 0.001
    assert list(X_tr.columns) == FEATURES
    assert X_tr.index.equals(y_tr.index) and X_val.index.equals(y_val.index)
    assert not X_tr.index.isin(X_val.index).any()
    assert not overlap_with_train(X_tr, X_val).any(), target

    share_tr = train_df.loc[X_tr.index, "attack_cat"].value_counts(normalize=True)
    share_val = train_df.loc[X_val.index, "attack_cat"].value_counts(normalize=True)
    assert (share_tr - share_val).abs().max() < 0.002, target

assert set(splits["label"][2].unique()) == {0, 1}
assert splits["attack_cat"][2].nunique() == 10

In [12]:
assert train_val_split(train_df, "label")[1].index.equals(splits["label"][1].index)
assert len(train_df) == 175341

## preprocessing.py

In [13]:
for target, (X_tr, X_val, y_tr, y_val) in splits.items():
    pre = build_preprocessor().fit(X_tr)
    names = list(pre.get_feature_names_out())
    Z_tr, Z_val, Z_test = (np.asarray(pre.transform(X)) for X in (X_tr, X_val, X_test))

    assert len(names) == Z_tr.shape[1] == Z_val.shape[1] == Z_test.shape[1] == 59, (target, Z_tr.shape)
    assert len(Z_test) == len(X_test)
    for Z in (Z_tr, Z_val, Z_test):
        assert np.isfinite(Z).all(), target

    num = Z_tr[:, :len(NUMERIC)]
    assert np.allclose(num.mean(axis=0), 0, atol=1e-6)
    assert np.allclose(num.std(axis=0), 1, atol=1e-6)

    for col in CATEGORICAL:
        idx = [i for i, n in enumerate(names) if n.startswith(f"cat__{col}_")]
        assert (Z_test[:, idx].sum(axis=1) == 1).all(), (target, col)

    unseen = X_test["state"].isin(["ACC", "CLO"]).to_numpy()
    assert unseen.sum() == 5
    assert (Z_test[unseen, names.index("cat__state_infrequent_sklearn")] == 1).all()

    assert np.array_equal(np.asarray(pre.transform(test_df)), Z_test)

In [14]:
print("all checks passed")

all checks passed
